In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/karishmas24/master-dataset/master_training_data.csv


In [15]:
import tensorflow as tf
import numpy as np
import pandas as pd
import xgboost as xgb
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
import matplotlib.pyplot as plt

In [16]:
# ==========================================
# 1. DATA LOADING & PREP
# ==========================================
print("Data Loading start")
file_path = '/kaggle/input/datasets/karishmas24/master-dataset/master_training_data.csv'
df = pd.read_csv(file_path)
print("Data Loading finish")

#drop unwanted columns
print("start clearing unwanted columns")
client_ids = df['client_id']
cols_to_drop = ['id', 'card_id', 'merchant_id']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

X = df.drop(columns=['is_fraud', 'client_id']).fillna(-1)
y = df['is_fraud']
print("unwamted columns clear")


print("start splitting my data")
X_train, X_test, y_train, y_test, client_train, client_test = train_test_split(
    X, y, client_ids, test_size=0.2, random_state=42, stratify=y
)
print("splitting complete")

Data Loading start
Data Loading finish
start clearing unwanted columns
unwamted columns clear
start splitting my data
splitting complete


In [17]:
# ==========================================
# 2. STAGE 1: XGBOOST BOUNCER
# ==========================================


print("\n2. Training STAGE 1: XGBoost on GPU...")
#weight tell us, how much xgboost need to secure to say its safe
stage1_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

stage1_model = xgb.XGBClassifier(
    n_estimators=150,        
    max_depth=6,             
    learning_rate=0.1,       
    scale_pos_weight=stage1_weight, 
    tree_method="hist",      
    device="cuda",           
    random_state=42
)
print("start training")
stage1_model.fit(X_train, y_train)
print("training completed")


y_probs_s1_train = stage1_model.predict_proba(X_train)[:, 1]
y_probs_s1_test = stage1_model.predict_proba(X_test)[:, 1]


precisions_s1, recalls_s1, thresholds_s1 = precision_recall_curve(y_test,y_probs_s1_test)
s1_safe_threshold = thresholds_s1[np.where(recalls_s1 >= 0.99)[0][-1]]
print(f"   -> Stage 1 High-Recall Threshold: {s1_safe_threshold:.4f}")


2. Training STAGE 1: XGBoost on GPU...
start training
training completed
   -> Stage 1 High-Recall Threshold: 0.0765


In [18]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

print("\n==================================================")
print("         DETAILED ML METRICS (PRECISION & RECALL) ")
print("==================================================")

y_pred_stage1 = (y_probs_s1_test >= s1_safe_threshold).astype(int)
precision = precision_score(y_test, y_pred_stage1)
recall = recall_score(y_test, y_pred_stage1)
f1 = f1_score(y_test, y_pred_stage1)

print(f"Precision : {precision:.4f} -> (Jab model ne 'Fraud' bola, toh wo kitni baar sach mein Fraud tha?)")
print(f"Recall    : {recall:.4f} -> (Total asali Frauds mein se, model ne kitne pakad liye?)")
print(f"F1-Score  : {f1:.4f} -> (Precision aur Recall ka perfect math balance)")

print("\n--- Full Classification Report ---")
print(classification_report(y_test, y_pred_stage1, target_names=['Safe (0)', 'Fraud (1)']))




         DETAILED ML METRICS (PRECISION & RECALL) 
Precision : 0.0106 -> (Jab model ne 'Fraud' bola, toh wo kitni baar sach mein Fraud tha?)
Recall    : 0.9902 -> (Total asali Frauds mein se, model ne kitne pakad liye?)
F1-Score  : 0.0211 -> (Precision aur Recall ka perfect math balance)

--- Full Classification Report ---
              precision    recall  f1-score   support

    Safe (0)       1.00      0.86      0.93   1780327
   Fraud (1)       0.01      0.99      0.02      2666

    accuracy                           0.86   1782993
   macro avg       0.51      0.93      0.47   1782993
weighted avg       1.00      0.86      0.92   1782993



In [19]:
def create_sequences(X_data, y_data, client_ids, lookback=5):
    X_seq, y_seq = [], []
    df_temp = pd.DataFrame(X_data)
    df_temp['is_fraud'] = y_data.values
    df_temp['client_id'] = client_ids.values
    df_temp = df_temp.sort_values(by=['client_id'])
    
    for client, group in df_temp.groupby('client_id'):
        if len(group) < lookback:
            continue
        features = group.drop(columns=['is_fraud', 'client_id']).values
        labels = group['is_fraud'].values
        for i in range(lookback, len(group) + 1):
            X_seq.append(features[i-lookback:i])
            y_seq.append(labels[i-1])
    return np.array(X_seq), np.array(y_seq)

In [20]:
# ==========================================
# 3. STAGE 2: LSTM PREP
# ==========================================
print("\n3. Preparing Temporal Sequences...")
LOOKBACK = 5

print('start filter----store doubtful and pass safeful')
s2_train_mask = y_probs_s1_train >= s1_safe_threshold
print('done')

scaler = StandardScaler()
X_train_s2_scaled = pd.DataFrame(scaler.fit_transform(X_train[s2_train_mask]), columns=X_train.columns)

X_train_lstm, y_train_lstm = create_sequences(
    X_train_s2_scaled, y_train[s2_train_mask], client_train[s2_train_mask], lookback=LOOKBACK
)


3. Preparing Temporal Sequences...
start filter----store doubtful and pass safeful
done


In [21]:
# ==========================================
# 4. STAGE 2: LSTM TRAINING
# ==========================================
print("\n4. Training STAGE 2: Deep LSTM on GPU...")

#Architecture
lstm_model = Sequential([
    Input(shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    LSTM(64, return_sequences=True),
    BatchNormalization(),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    BatchNormalization(),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
lstm_model.fit(
    X_train_lstm, y_train_lstm,
    epochs=20,         
    batch_size=2048,    
    verbose=1
)


4. Training STAGE 2: Deep LSTM on GPU...
Epoch 1/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - AUC: 0.5159 - loss: 0.1211
Epoch 2/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - AUC: 0.7004 - loss: 0.0567
Epoch 3/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - AUC: 0.7996 - loss: 0.0522
Epoch 4/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - AUC: 0.8274 - loss: 0.0498
Epoch 5/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - AUC: 0.8423 - loss: 0.0483
Epoch 6/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - AUC: 0.8505 - loss: 0.0472
Epoch 7/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - AUC: 0.8595 - loss: 0.0459
Epoch 8/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - AUC: 0.8644 - loss: 0.0451
Epoch 9/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - AUC: 0.8695 - loss: 0.0442
Epoch 10/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - AUC: 0.8744 - loss: 0.0436
Epoch 11/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - AUC: 0.8770 - loss: 0.0429
Epoch 12/20
483/483 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step 

In [22]:
# LSTM Evaluation
s2_test_mask = y_probs_s1_test >= s1_safe_threshold

X_test_s2_scaled = pd.DataFrame(scaler.transform(X_test[s2_test_mask]), columns=X_test.columns)

X_test_lstm, y_test_lstm = create_sequences(

    X_test_s2_scaled, y_test[s2_test_mask], client_test[s2_test_mask], lookback=LOOKBACK

)


y_probs_s2 = lstm_model.predict(X_test_lstm).flatten()

print("start auto rejection")
precisions_s2, recalls_s2, thresholds_s2 = precision_recall_curve(y_test_lstm, y_probs_s2)
s2_reject_threshold = thresholds_s2[np.where(precisions_s2 >= 0.85)[0][0]]
print("auto Reject completed")
print(f"-> Stage 2 Auto-Reject Threshold: {s2_reject_threshold:.4f}")

7603/7603 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step
start auto rejection
auto Reject completed
-> Stage 2 Auto-Reject Threshold: 0.7391


In [23]:
#==========================================
# 5. FINAL EXACT DASHBOARD
# ==========================================
print("\n==================================================")
print("         SYSTEM LEVEL ROUTING BREAKDOWN           ")
print("==================================================")
total_requests = len(X_test)

s1_safe_mask = y_probs_s1_test < s1_safe_threshold
stage1_approved_count = s1_safe_mask.sum()

s2_reject_mask = y_probs_s2 >= s2_reject_threshold
stage2_rejected_count = s2_reject_mask.sum()

routed_to_human = total_requests - stage1_approved_count - stage2_rejected_count
automation_rate = ((stage1_approved_count + stage2_rejected_count) / total_requests) * 100


         SYSTEM LEVEL ROUTING BREAKDOWN           


In [24]:
# Throughput calculation
start_time = time.time()
_ = stage1_model.predict_proba(X_test)
if len(X_test_lstm) > 0:
    _ = lstm_model.predict(X_test_lstm, batch_size=2048, verbose=0)
end_time = time.time()
processing_time = end_time - start_time
throughput = total_requests / processing_time if processing_time > 0 else 0

s1_false_negatives = np.sum((s1_safe_mask == True) & (y_test == 1))
s2_false_positives = np.sum((s2_reject_mask == True) & (y_test_lstm == 0))

print(f"Total Traffic Processed       : {total_requests:,}")
print(f"Stage 1 Auto-Approved         : {stage1_approved_count:,} ({(stage1_approved_count/total_requests)*100:.2f}%)")
print(f"Stage 2 Auto-Rejected         : {stage2_rejected_count:,} ({(stage2_rejected_count/total_requests)*100:.2f}%)")
print(f"Routed to React Dashboard     : {routed_to_human:,} ({(routed_to_human/total_requests)*100:.2f}%)")
print(f"Total System Automation Rate  : {automation_rate:.2f}%")
print(f"Total Processing Throughput   : {throughput:.2f} tx/sec")
print("==================================================")
print("         AUTOMATED BLOCKING ERRORS (PRE-HITL)     ")
print("==================================================")
print(f"Hard False Negatives (S1 Missed Fraud) : {s1_false_negatives}")
print(f"Hard False Positives (S2 False Blocks) : {s2_false_positives}")

Total Traffic Processed       : 1,782,993
Stage 1 Auto-Approved         : 1,534,851 (86.08%)
Stage 2 Auto-Rejected         : 423 (0.02%)
Routed to React Dashboard     : 247,719 (13.89%)
Total System Automation Rate  : 86.11%
Total Processing Throughput   : 795892.52 tx/sec
         AUTOMATED BLOCKING ERRORS (PRE-HITL)     
Hard False Negatives (S1 Missed Fraud) : 26
Hard False Positives (S2 False Blocks) : 63


In [25]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

print("\n==================================================")
print("     MATH TO CODE: CONFUSION MATRIX & METRICS     ")
print("==================================================")

y_pred_s1 = (y_probs_s1_test >= s1_safe_threshold).astype(int)

# 2. Confusion Matrix--- TP, FP, FN, TN 
# .ravel() function 2x2 matrix ko khol kar 4 variables mein daal deta hai
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_s1).ravel()

print("--- The 4 Pillars (Stage 1 Bouncer) ---")
print(f"True Positives (TP)  : {tp} (Sahi pakde gaye Frauds)")
print(f"False Positives (FP) : {fp} (doubt tha par Shareef hai)")
print(f"False Negatives (FN) : {fn} (Missed Frauds / Bouncer se bach nikle)")
print(f"True Negatives (TN)  : {tn} (Sahi pass hue Shareef log)")

# 3. Precision, Recall aur F1-Score calculate karna
precision = precision_score(y_test, y_pred_s1)
recall = recall_score(y_test, y_pred_s1)
f1 = f1_score(y_test, y_pred_s1)

print("\n--- The Final Formulas Output ---")
print(f"Precision : {precision:.4f}  [Formula: TP / (TP + FP)]")
print(f"Recall    : {recall:.4f}  [Formula: TP / (TP + FN)]")
print(f"F1-Score  : {f1:.4f}  [Formula: Balance of Precision & Recall]")
print("==================================================")


     MATH TO CODE: CONFUSION MATRIX & METRICS     
--- The 4 Pillars (Stage 1 Bouncer) ---
True Positives (TP)  : 2640 (Sahi pakde gaye Frauds)
False Positives (FP) : 245502 (doubt tha par Shareef hai)
False Negatives (FN) : 26 (Missed Frauds / Bouncer se bach nikle)
True Negatives (TN)  : 1534825 (Sahi pass hue Shareef log)

--- The Final Formulas Output ---
Precision : 0.0106  [Formula: TP / (TP + FP)]
Recall    : 0.9902  [Formula: TP / (TP + FN)]
F1-Score  : 0.0211  [Formula: Balance of Precision & Recall]


In [28]:

import joblib

# 1. XGBoost save
stage1_model.save_model("stage1_xgb.json")

# 2. LSTM save
lstm_model.save("stage2_lstm.keras")

# 3. Scaler save
joblib.dump(scaler, "scaler.pkl")

print("Pipeline Saved! Files: stage1_xgb.json, stage2_lstm.keras, scaler.pkl")

Pipeline Saved! Files: stage1_xgb.json, stage2_lstm.keras, scaler.pkl


In [29]:
from IPython.display import FileLink

# Har file ke liye download link generate karein
display(FileLink('stage1_xgb.json'))
display(FileLink('stage2_lstm.keras'))
display(FileLink('scaler.pkl'))

/kaggle/working/stage1_xgb.json

/kaggle/working/stage2_lstm.keras

/kaggle/working/scaler.pkl